Группируем транзакции клиента по месяцам, убираем неинформативных клиентов, проводим первичную сегментацию, добавляем признаки, выделяем target = кол-во транзакций в след. месяце

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_parquet(
    "../data/test_datasets/mbd_dataset/detail/trx/fold=0"
)

# Сначала вычислим признаки, которые используют другие валюты, кроме одинадцатой
Потом их удалим и будем считать признаки связанные с сумой транзакций в 11 валюте по месяцам

In [4]:
# Создаём признаки по валютам для каждого клиента
# 1. Количество уникальных валют
# 2. Доля транзакций в валюте 11 (в eda выяснили, что она основная)
# 3. Флаг мультивалютности

currency_features = df.groupby("client_id").agg(
    n_unique_currency=("currency", "nunique"),  # количество уникальных валют
    currency_11_count=("currency", lambda x: (x == 11).sum()),  # количество транзакций в валюте 11
    total_tx=("currency", "count")  # общее количество транзакций
).reset_index()

# Доля транзакций в валюте 11
currency_features["share_currency_11"] = currency_features["currency_11_count"] / currency_features["total_tx"]

# Флаг мультивалютности
currency_features["is_multicurrency"] = (currency_features["n_unique_currency"] > 1).astype(int)

# Оставляем только нужные признаки
currency_features = currency_features[[
    "client_id",
    "n_unique_currency",
    "share_currency_11",
    "is_multicurrency"
]]

print("Признаки по валютам на уровне клиента (первые строки)")
display(currency_features.head())
display(currency_features.describe())


Признаки по валютам на уровне клиента (первые строки)


,client_id,n_unique_currency,share_currency_11,is_multicurrency
0,00098f117ba54c5f21436d0687943b7140c356299f64cf...,1,1.0,0
1,000afc90f17bbfcfe559432c975a1f7b65fd8c27ba222c...,1,1.0,0
2,000e047a31e50ba35f71c81962b4eb0b9a2d6080cf23a1...,1,1.0,0
3,00112e0ce2b5d48c4e83ced4c0d01b8941c9f42177be81...,1,1.0,0
4,0011eb856025cc73656723ea7af19e335cdb2f005f2eac...,1,1.0,0


,n_unique_currency,share_currency_11,is_multicurrency
count,20032.000000,20032.000000,20032.000000
mean,1.017023,0.999631,0.013029
std,0.160374,0.005109,0.113402
min,1.000000,0.800000,0.000000
25%,1.000000,1.000000,0.000000
50%,1.000000,1.000000,0.000000
75%,1.000000,1.000000,0.000000
max,5.000000,1.000000,1.000000


Получили

currency_features

In [5]:
# Event type признаки 
# Считаем количество транзакций каждого типа для каждого клиента
event_counts = df.groupby(["client_id", "event_type"])["amount"].count().unstack(fill_value=0)

# Преобразуем в долю от всех транзакций клиента
event_shares = event_counts.div(event_counts.sum(axis=1), axis=0)

# Берем топ-10 типов событий и объединяем все остальные в "other_events"
top_events = event_shares.iloc[:, :10].copy()
top_events["other_events"] = 1 - top_events.sum(axis=1)

# Получаем таблицу с client_id и долями событий
event_features = top_events.reset_index()


# src/dst признаки
src_cols = ["src_type11", "src_type12", "src_type21", "src_type22", "src_type31", "src_type32"]
dst_cols = ["dst_type11", "dst_type12"]

src_dst_features = pd.DataFrame({"client_id": df["client_id"].unique()})

for col in src_cols + dst_cols:
    # Считаем количество уникальных значений по каждому клиенту
    stats = df.groupby("client_id")[col].nunique().rename(f"{col}_n_unique").reset_index()
    # Объединяем с таблицей src_dst_features
    src_dst_features = src_dst_features.merge(stats, on="client_id", how="left")


In [6]:
display(event_features.head())
display(src_dst_features.head())


event_type,client_id,1,3,4,5,6,7,8,9,10,11,other_events
0,00098f117ba54c5f21436d0687943b7140c356299f64cf...,1.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000
1,000afc90f17bbfcfe559432c975a1f7b65fd8c27ba222c...,0.407563,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.592437
2,000e047a31e50ba35f71c81962b4eb0b9a2d6080cf23a1...,0.662371,0.0,0.002577,0.0,0.0,0.002577,0.0,0.0,0.0,0.012887,0.319588
3,00112e0ce2b5d48c4e83ced4c0d01b8941c9f42177be81...,0.930233,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.011628,0.058140
4,0011eb856025cc73656723ea7af19e335cdb2f005f2eac...,0.120370,0.0,0.002315,0.0,0.0,0.004630,0.0,0.0,0.0,0.113426,0.759259


,client_id,src_type11_n_unique,src_type12_n_unique,src_type21_n_unique,src_type22_n_unique,src_type31_n_unique,src_type32_n_unique,dst_type11_n_unique,dst_type12_n_unique
0,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2,2,1,1,1,1,3,4
1,05d943d80444604f2450e0cf4bd4d11524f065fee2e837...,1,1,1,1,1,1,3,3
2,38708d828e036ac1adefb5836a27873988e2105cd7d560...,1,1,1,1,1,1,2,2
3,40971e454cfb41e4de3a059dc724b333445a7224e9a6a4...,4,4,1,1,1,1,5,8
4,5019c941292eecc348d34349c82f2ae84323953b419c91...,5,9,1,1,1,1,9,13


Получили

src_dst_features

event_features

In [ ]:
print(f"Было транзакций: {len(df):,}")
# Оставляем только транзакции в основной валюте
df = df[df["currency"] == 11].copy()

print(f"Оставлено транзакций в валюте 11: {len(df):,}")
print(f"Количество клиентов: {df['client_id'].nunique():,}")


Оставлено транзакций в валюте 11: 7,626,656
Количество клиентов: 20,032


# Фильтрация клиентов c недостаточной историей

In [8]:
# Рассматриваем только клиентов с историей ≥6 месяцев и количеством транзакций ≥20
client_stats = df.groupby("client_id").agg(
    tx_count=("amount", "count"),
    history_months=("event_time", lambda x: x.dt.to_period("M").nunique())
).reset_index()

# фильтр
eligible_clients = client_stats[
    (client_stats["tx_count"] >= 20) & 
    (client_stats["history_months"] >= 6)
]["client_id"]

df = df[df["client_id"].isin(eligible_clients)]
print(f"Оставлено клиентов: {df['client_id'].nunique()}")

Оставлено клиентов: 16482


# Первичная сегментация клиентов

In [10]:
# Суммарная сумма транзакций по клиентам
client_amounts = df.groupby("client_id")["amount"].sum().reset_index()

# Определяем порог для крупных клиентов
threshold_large = client_amounts["amount"].quantile(0.99)

# Добавляем сегмент
client_amounts["client_segment"] = np.where(client_amounts["amount"] > threshold_large, 
                                            "Крупные", "Массовые")

# Сколько клиентов в каждом сегменте
print(client_amounts["client_segment"].value_counts())


client_segment
Массовые    16317
Крупные       165
Name: count, dtype: int64


# Числовые агрегаты по транзакциям:

In [11]:
# Агрегация по клиентам: считаем основные статистики по суммам транзакций
amount_client = df.groupby("client_id")["amount"].agg(
    tx_count="count",   # общее количество транзакций клиента
    total_sum="sum",    # общая сумма транзакций
    mean="mean",        # средний чек
    median="median",    # медианный чек
    std="std",          # стандартное отклонение сумм
    min="min",          # минимальная сумма транзакции
    max="max"           # максимальная сумма транзакции
).reset_index()

# Добавляем квантили сумм транзакций для каждого клиента
quantiles = df.groupby("client_id")["amount"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()

# Переименовываем столбцы квантилей в читаемый формат
quantiles.columns = [f"q_{int(q*100)}" for q in quantiles.columns]

# Объединяем квантили с основной таблицей агрегированных признаков
amount_client = amount_client.join(quantiles, on="client_id")


In [12]:
# Преобразуем дату события в период "год-месяц"
df["year_month"] = df["event_time"].dt.to_period("M")

# Считаем суммарную сумму транзакций по каждому клиенту и по каждому месяцу
monthly_sum = df.groupby(["client_id", "year_month"])["amount"].sum()

# Агрегируем месячные суммы на уровне клиента:
monthly_features = monthly_sum.groupby("client_id").agg(
    monthly_sum_mean="mean",   # средняя месячная сумма
    monthly_sum_std="std",     # разброс (вариативность) по месяцам
    monthly_sum_max="max"      # максимальная сумма за месяц
)

# Объединяем агрегированные месячные признаки с основной таблицей признаков по клиенту
amount_client = amount_client.join(monthly_features, on="client_id")


In [13]:
# Суммы транзакций по клиенту и месяцу
monthly_sum = df.groupby(["client_id", "year_month"])["amount"].sum().sort_index().reset_index()

# Создаем таблицу с признаками по клиенту и месяцу
monthly_features = monthly_sum.copy()

# Лаговые признаки
monthly_features["amount_lag_1"] = monthly_features.groupby("client_id")["amount"].shift(1)  # сумма за предыдущий месяц
monthly_features["amount_lag_3"] = monthly_features.groupby("client_id")["amount"].shift(3)  # сумма за 3 месяца назад
monthly_features["amount_lag_6"] = monthly_features.groupby("client_id")["amount"].shift(6)  # сумма за 6 месяцев назад

# Считаем количество транзакций по клиенту и месяцу
monthly_count = df.groupby(["client_id", "year_month"])["amount"].count().sort_index().reset_index(name="tx_count")

# Лаговые признаки для количества транзакций
monthly_features = monthly_features.merge(monthly_count, on=["client_id", "year_month"])
monthly_features["count_lag_1"] = monthly_features.groupby("client_id")["tx_count"].shift(1)
monthly_features["count_lag_3"] = monthly_features.groupby("client_id")["tx_count"].shift(3)
monthly_features["count_lag_6"] = monthly_features.groupby("client_id")["tx_count"].shift(6)

# Скользящие средние сумм транзакций для выявления трендов
monthly_features["amount_roll_3m"] = monthly_features.groupby("client_id")["amount"].transform(lambda x: x.rolling(3, min_periods=1).mean())
monthly_features["amount_roll_6m"] = monthly_features.groupby("client_id")["amount"].transform(lambda x: x.rolling(6, min_periods=1).mean())

# monthly_features содержит клиентские признаки с лагами и скользящими средними


# Объединим все признаки и добавим таргет

In [16]:
# Берем за основу уже рассчитанные месячные признаки
client_month = monthly_features.copy()

# Переименуем столбцы для согласованности
client_month = client_month.rename(columns={"amount": "amount_sum", "tx_count": "tx_count"})

# Добавляем клиентские признаки (агрегаты по суммам)
client_month = client_month.merge(amount_client, on="client_id", how="left")

# Добавляем валютные признаки
client_month = client_month.merge(currency_features, on="client_id", how="left")

# Добавляем доли событий
client_month = client_month.merge(event_features, on="client_id", how="left")

# Добавляем src/dst признаки
client_month = client_month.merge(src_dst_features, on="client_id", how="left")

# Создаем таргет — сумма транзакций в следующем месяце
client_month["amount_next_month"] = client_month.groupby("client_id")["amount_sum"].shift(-1)

# Убираем строки без таргета
client_month = client_month.dropna(subset=["amount_next_month"])

# Сохраняем финальный датасет
client_month.to_parquet("../data/processed/client_month_features.parquet", index=False)

print(f"Финальный датасет: {client_month.shape[0]} строк, {client_month.shape[1]} признаков")


Финальный датасет: 332605 строк, 50 признаков


In [17]:
client_month.info()

<class 'pandas.core.frame.DataFrame'>
Index: 332605 entries, 0 to 349085
Data columns (total 50 columns):
 #   Column               Non-Null Count   Dtype    
---  ------               --------------   -----    
 0   client_id            332605 non-null  object   
 1   year_month           332605 non-null  period[M]
 2   amount_sum           332605 non-null  float32  
 3   amount_lag_1         316123 non-null  float32  
 4   amount_lag_3         283159 non-null  float32  
 5   amount_lag_6         233862 non-null  float32  
 6   tx_count_x           332605 non-null  int64    
 7   count_lag_1          316123 non-null  float64  
 8   count_lag_3          283159 non-null  float64  
 9   count_lag_6          233862 non-null  float64  
 10  amount_roll_3m       332605 non-null  float64  
 11  amount_roll_6m       332605 non-null  float64  
 12  tx_count_y           332605 non-null  int64    
 13  total_sum            332605 non-null  float32  
 14  mean                 332605 non-null  flo